In [1]:
from operator import itemgetter
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings


C:\Users\ARTI\AppData\Local\Temp\ipykernel_21820\1258926756.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
BASE_DIR = Path.cwd()
FAISS_DIR = BASE_DIR / "faiss_index"

EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "mistral"
OLLAMA_BASE_URL = "http://localhost:11434"

TOP_K = 3
MAX_HISTORY_MESSAGES = 6    # remember the last 3 question-and-answer pairs

print("Index folder exists:", FAISS_DIR.exists())


Index folder exists: True


In [3]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

store = FAISS.load_local(
    str(FAISS_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
    normalize_L2=True,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)

retriever = store.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})

llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)


def format_docs(docs) -> str:
    return "\n\n---\n\n".join(d.page_content for d in docs)


print("Vectors loaded:", store.index.ntotal)


c:\Users\ARTI\Desktop\Abhishek\RAG-Chatbot-2-FAISS\.venv\Lib\site-packages\langchain_community\vectorstores\faiss.py:233: UserWarning: Normalizing L2 is not applicable for metric type: DistanceStrategy.MAX_INNER_PRODUCT
  warnings.warn(


Vectors loaded: 20


In [4]:
def preview(text: str, n: int = 70) -> str:
    text = text.replace("\x7f", "-")      # PDF bullet glyph -> normal dash
    text = " ".join(text.split())         # newlines and extra spaces -> single space
    return text[:n] + ("..." if len(text) > n else "")


def show(title: str, query: str) -> None:
    print(f"--- {title} ---")
    print(f"Query: {query}")
    for d in retriever.invoke(query):
        print(f"  chunk {d.metadata['chunk_id']:>2} | page {d.metadata['page'] + 1} | {preview(d.page_content)}")
    print()


show("Standalone question", "Why is chunking useful in RAG?")
show("Vague follow-up (no memory)", "Why is it useful?")


--- Standalone question ---
Query: Why is chunking useful in RAG?
  chunk  7 | page 4 | RAG Notes (updated) Page 4 Step 3. Find relevant chunks - Compare the ...
  chunk  9 | page 4 | - A fallback when no relevant chunk is found ("I don't know"). Text sp...
  chunk 10 | page 5 | RAG Notes (updated) Page 5 More to try: - Semantic chunking - Markdown...

--- Vague follow-up (no memory) ---
Query: Why is it useful?
  chunk 12 | page 5 | - create_history_aware_retriever() - create_stuff_documents_chain() No...
  chunk  2 | page 2 | - Give the retrieved information to the LLM. - The LLM answers based o...
  chunk 11 | page 5 | - Pinecone - Weaviate - Qdrant - Milvus - DataStax Astra DB More to tr...



In [5]:
# Prompt 1: rewrite a follow-up into a full question
contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given the chat history and the latest user question, rewrite the "
            "question so it can be understood on its own. "
            "Replace words like 'it', 'that' or 'they' with what they refer to. "
            "Do NOT answer the question. Return ONLY the rewritten question. "
            "If it already makes sense on its own, return it unchanged.",
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)


In [6]:


# Prompt 2: answer using the chunks and the history
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer questions about RAG (Retrieval-Augmented Generation) "
            "using ONLY the context below.\n\n"
            "Rules:\n"
            "1. Answer ONLY what the question asks. No extra background or related topics.\n"
            "2. The context may contain unrelated parts. Ignore them.\n"
            "3. Be short: 2 to 4 sentences. Use a short bullet list only for steps, types or comparisons.\n"
            "4. Use the chat history only to understand what the user refers to. Never answer from it.\n"
            "5. If the answer is not in the context, reply exactly: \"I couldn't find that in the notes.\"\n\n"
            "Context:\n{context}",
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)

print("Prompt 1 needs:", contextualize_prompt.input_variables)
print("Prompt 2 needs:", answer_prompt.input_variables)


Prompt 1 needs: ['chat_history', 'question']
Prompt 2 needs: ['chat_history', 'context', 'question']


In [7]:
def create_conversational_rag(retriever, llm):
    # Small chain: history + follow-up -> full question
    contextualize_chain = contextualize_prompt | llm | StrOutputParser()

    def rewrite_question(inputs: dict) -> str:
        if not inputs["chat_history"]:
            return inputs["question"]          # first question: nothing to rewrite
        return contextualize_chain.invoke(inputs)

    return (
        RunnablePassthrough.assign(
            standalone_question=RunnableLambda(rewrite_question)
        )
        | RunnablePassthrough.assign(
            context=itemgetter("standalone_question") | retriever | format_docs
        )
        | answer_prompt
        | llm
    )


conversational_rag = create_conversational_rag(retriever, llm)
print("Conversational RAG chain created")


Conversational RAG chain created


In [8]:
chat_history = []


def chat(question: str) -> None:
    print("You:", question)
    print("Bot: ", end="", flush=True)

    answer = ""
    for chunk in conversational_rag.stream(
        {"question": question, "chat_history": chat_history}
    ):
        print(chunk.content, end="", flush=True)
        answer += chunk.content
    print("\n")

    # Remember this turn
    chat_history.extend(
        [HumanMessage(content=question), AIMessage(content=answer)]
    )

    # Keep only the last MAX_HISTORY_MESSAGES messages
    del chat_history[:-MAX_HISTORY_MESSAGES]
    

In [9]:
chat_history.clear()

chat("What is FAISS?")
chat("Is it a vector store or a vector database?")
chat("Is loading it safe?")


You: What is FAISS?
Bot:  FAISS is a library for efficient similarity search and clustering of dense vectors. It is used in the context of Retrieval-Augmented Generation (RAG) for storing vectors of text chunks in a database.

You: Is it a vector store or a vector database?
Bot:  The context mentions both a vector store and a vector database. A vector store is a simple library for storing vectors in memory or as a local file, with basic search capabilities (examples: FAISS, Chroma, Annoy, ScaNN). A vector database, on the other hand, offers advanced search with metadata filters, CRUD operations, distributed and scalable capabilities (examples: Pinecone, Weaviate, Qdrant, Milvus, DataStax Astra DB).

You: Is loading it safe?
Bot:  The context mentions that FAISS.load_local() needs allow_dangerous_deserialization=True and only to load index files created by yourself. However, it does not explicitly state whether loading it is safe or not. To ensure safety, it is recommended to follow bes

In [10]:
chat_history.clear()
chat("What is FAISS?")
chat("How do I save it?")

for m in chat_history:
    print(f"{type(m).__name__:13} -> {m.content[:60]!r}")
    

You: What is FAISS?
Bot:  FAISS is a library for efficient similarity search and clustering of dense vectors. It is used in the context of Retrieval-Augmented Generation (RAG) for storing vectors of text chunks in a database.

You: How do I save it?
Bot:  To save the FAISS index, you can use the `FAISS.save_local()` method. Here's a short summary of the steps:

1. Embed the chunks into vectors using an embedding model.
2. Store the vectors in a FAISS index.
3. Save the index using `FAISS.save_local()`.

It's important to note that you should only load index files you created yourself, and when loading the index, set `allow_dangerous_deserialization=True`. This helps ensure data privacy and security.

HumanMessage  -> 'What is FAISS?'
AIMessage     -> ' FAISS is a library for efficient similarity search and clus'
HumanMessage  -> 'How do I save it?'
AIMessage     -> ' To save the FAISS index, you can use the `FAISS.save_local('
